In [ ]:
import sys, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U",
                "sentence-transformers", "xl-durel-utils", "scikit-learn", "scipy", "bcubed==1.5"], check=True)
import torch
assert torch.cuda.is_available(), "Select an L4 runtime."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json, re, time, numpy as np, torch
from collections import Counter
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from xl_durel_utils import tokenize_truncate_decode
from sklearn.cluster import AgglomerativeClustering
from project_utils import new_run, save_json, record_environment
from semeval_task3.data import load_dataset

dev = load_dataset(PROJECT / "work/data/dev")          # inputs only, no labels
assert not any(u.span_flags for u in dev.usages)
usages = list(dev.usages)

MODEL_ID = "sachinn1/xl-durel"
REV = HfApi().model_info(MODEL_ID).sha
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REV)
model = SentenceTransformer(MODEL_ID, revision=REV, device="cuda")
RUN = new_run(PROJECT, "step05_usage_groups")
print("XL-DURel revision:", REV, "| run:", RUN)

# Mark the target as "<t>word </t>" (the model's expected format), keeping long sentences centred on the target.
def tag(u):
    s = tokenize_truncate_decode(u.text, [u.start, u.end], tokenizer=tok, max_seq_len=128)
    m = re.search(r"<t>(.*?)</t>", s)
    if m and m.group(1).strip() == u.text[u.start:u.end].strip():
        return s, "helper"
    return u.text[:u.start] + "<t>" + u.text[u.start:u.end] + " </t>" + u.text[u.end:], "manual"

tagged, how = zip(*[tag(u) for u in usages])
print("Tagging:", Counter(how), "| example:", tagged[0][:160])

t0 = time.time()
emb = model.encode(list(tagged), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(f"Embedded {len(emb)} sentences in {time.time() - t0:.0f}s")
np.save(RUN / "xl_durel_embeddings.npy", emb)
save_json(RUN / "usage_keys.json", [list(u.key) for u in usages])

# How far apart are sentences of the same word? (calibrates the grouping thresholds)
words = sorted({u.word for u in usages})
idx = {w: [i for i, u in enumerate(usages) if u.word == w] for w in words}
d_all = np.concatenate([(1 - emb[ix] @ emb[ix].T)[np.triu_indices(len(ix), 1)] for ix in idx.values()])
print("Within-word cosine distance percentiles 10/50/90:", np.percentile(d_all, [10, 50, 90]).round(3))

# Group each word's sentences at several levels of detail (larger threshold = fewer, coarser groups)
THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
clusters = {}
for t in THRESHOLDS:
    clusters[t] = {}
    for w in words:
        ix = idx[w]
        lab = AgglomerativeClustering(n_clusters=None, distance_threshold=t,
                                      metric="cosine", linkage="average").fit_predict(emb[ix])
        for i, c in zip(ix, lab):
            clusters[t]["|".join(usages[i].key)] = int(c)
save_json(RUN / "clusters.json", {str(t): v for t, v in clusters.items()})

print("\nNumber of groups per word at each threshold:")
print(f"{'word':<10}" + "".join(f"{t:>6}" for t in THRESHOLDS))
for w in words:
    row = [len({clusters[t]['|'.join(usages[i].key)] for i in idx[w]}) for t in THRESHOLDS]
    print(f"{w:<10}" + "".join(f"{n:>6}" for n in row))

# Peek at the groups at one middle threshold: sizes + 2 sentences nearest each group's centre (no labels)
SHOW_T = 0.5
print(f"\nExample groups at threshold {SHOW_T} (largest 4 per word):")
for w in words:
    ix = idx[w]
    lab = np.array([clusters[SHOW_T]["|".join(usages[i].key)] for i in ix])
    print(f"\n=== {w}: {len(set(lab))} groups, sizes {sorted(Counter(lab).values(), reverse=True)[:8]}")
    for c, n in Counter(lab).most_common(4):
        members = [ix[j] for j in np.where(lab == c)[0]]
        centre = emb[members].mean(0)
        nearest = sorted(members, key=lambda i: -float(emb[i] @ centre))[:2]
        print(f"  group {c} (n={n}):")
        for i in nearest:
            print("     ", usages[i].marked_text()[:150])

record_environment(PROJECT, RUN / "manifest.json", purpose="XL-DURel usage embeddings and label-free clustering",
                   model_id=MODEL_ID, model_revision=REV, thresholds=THRESHOLDS)

In [ ]:
import json, numpy as np
from collections import Counter, defaultdict
from sklearn.metrics import f1_score
from semeval_task3.data import load_dataset
from semeval_task3.evaluation import score, read_submission

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

section("FEASIBILITY CHECK: what is the best this grouping method could possibly do?")
print("Question: if every group were labelled perfectly (with its true majority label),")
print("would the result beat Gemma? This is a CEILING, not a real result. Real systems")
print("only reach part of it. Dev labels are used only to decide whether to build the method.")

cl_files = sorted((PROJECT / "outputs/runs/step05_usage_groups").glob("*/clusters.json"))
print(f"\nCluster file used : {cl_files[-1]}")
CL = json.loads(cl_files[-1].read_text())
TS = sorted(CL, key=float)

dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = dev.require_gold()
GEMMA_ZIP = PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a/gemma_raw.zip"
gemma = score(gold, read_submission(GEMMA_ZIP, dev.keys))
gw = {r["word"]: r["f1"] for r in gemma["per_word"]}
print(f"Gemma v1 reference: {gemma['average']:.4f} mean per-word F1")

words = sorted({u.word for u in dev.usages})
by_word = {w: [u for u in dev.usages if u.word == w] for w in words}

def oracle(w, t):
    us = by_word[w]
    groups = defaultdict(list)
    for u in us:
        groups[CL[t]["|".join(u.key)]].append(gold[u.key])
    maj = {g: int(np.mean(v) > 0.5) for g, v in groups.items()}
    y = [gold[u.key] for u in us]
    p = [maj[CL[t]["|".join(u.key)]] for u in us]
    return f1_score(y, p, zero_division=0), len(groups)

section("PER-WORD CEILING (cell = groups : best possible F1)")
print("Compare each cell with Gemma's F1 in the first column. Only thresholds with")
print("few groups (about 6 or fewer per word) are realistic for group-level judgment.\n")
print(f"{'word':<10}{'Gemma':>7}" + "".join(f"{'t=' + t:>12}" for t in TS))
res = {t: {} for t in TS}
for w in words:
    cells = []
    for t in TS:
        f, k = oracle(w, t)
        res[t][w] = (f, k)
        mark = "*" if f > gw[w] + 0.02 else " "
        cells.append(f"{k:>3}:{f:.3f}{mark}")
    print(f"{w:<10}{gw[w]:>7.3f}" + "".join(f"{c:>12}" for c in cells))
print("\n* = ceiling beats Gemma on that word by more than 0.02")

section("SUMMARY BY THRESHOLD")
print(f"{'threshold':<11}{'avg groups':>11}{'ceiling alone':>15}{'best of both*':>15}{'words helped':>14}")
for t in TS:
    f_or = np.mean([res[t][w][0] for w in words])
    f_best = np.mean([max(res[t][w][0], gw[w]) for w in words])
    k_avg = np.mean([res[t][w][1] for w in words])
    helped = sum(res[t][w][0] > gw[w] + 0.02 for w in words)
    print(f"{t:<11}{k_avg:>11.1f}{f_or:>15.3f}{f_best:>15.3f}{helped:>11}/10")
print("\n* 'best of both' = per word, the better of Gemma and the perfect-group ceiling.")
print("  It is an optimistic upper bound for any Gemma + groups combination.")

section("HOW TO READ THIS")
print("Focus on thresholds 0.5 to 0.7 (few groups, so group-level judgment is realistic).")
print(f" - If 'best of both' there is below ~{gemma['average'] + 0.04:.2f}: the method cannot")
print("   realistically gain enough. STOP and save the A100 time.")
print(" - If it is clearly higher and several words are helped: continue to checkpoint 2.")

In [ ]:
import sys, json, numpy as np
from pathlib import Path
from collections import defaultdict
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn", "scipy", "bcubed==1.5"], check=True)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap
from project_utils import read_scores, frozen, save_evaluation, save_json, new_run, record_environment

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

dev = load_dataset(PROJECT / "work/data/dev", with_gold=True); gold = dev.require_gold()
CL = json.loads(sorted((PROJECT / "outputs/runs/step05_usage_groups").glob("*/clusters.json"))[-1].read_text())
g = read_scores(PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json", dev)
v1 = score(gold, read_submission(PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a/gemma_raw.zip", dev.keys))
assert abs(v1["average"] - 0.6782392862783919) < 1e-10
gw = {r["word"]: r["f1"] for r in v1["per_word"]}
words = sorted({u.word for u in dev.usages})
by_word = {w: [u for u in dev.usages if u.word == w] for w in words}
TS = sorted(CL, key=float)
RUN = new_run(PROJECT, "step05_usage_groups_corrected")

# ---------------------------------------------------------------------------
section("PART 1: CORRECTED CEILING (F1-optimal choice of which groups to call YES)")
print("Previous version gave each group its majority label (maximises accuracy, not F1). Fixed here:")
print("groups are sorted by their share of YES answers and the best cut is chosen for F1.")
print("Still a ceiling (uses labels), used only to correct the record.\n")
def best_group_f1(w, t):
    groups = defaultdict(lambda: [0, 0])
    for u in by_word[w]:
        c = groups[CL[t]["|".join(u.key)]]; c[0] += gold[u.key]; c[1] += 1
    P = sum(p for p, n in groups.values())
    best, tp, pred = 0.0, 0, 0
    for p, n in sorted(groups.values(), key=lambda x: -x[0] / x[1]):
        tp += p; pred += n
        best = max(best, 2 * tp / (pred + P) if pred + P else 0)
    return best, len(groups)
print(f"{'word':<10}{'Gemma':>7}" + "".join(f"{'t=' + t:>12}" for t in TS))
ceil = {t: {} for t in TS}
for w in words:
    cells = []
    for t in TS:
        f, k = best_group_f1(w, t); ceil[t][w] = f
        cells.append(f"{k:>3}:{f:.3f}{'*' if f > gw[w] + 0.02 else ' '}")
    print(f"{w:<10}{gw[w]:>7.3f}" + "".join(f"{c:>12}" for c in cells))
print("\n* = group ceiling beats Gemma on that word by more than 0.02")
print(f"\n{'threshold':<11}{'mean ceiling':>13}{'words above Gemma':>19}")
for t in TS:
    print(f"{t:<11}{np.mean(list(ceil[t].values())):>13.3f}{sum(ceil[t][w] > gw[w] + 0.02 for w in words):>16}/10")

# ---------------------------------------------------------------------------
section("PART 2: REAL HYBRID: Gemma score blended with its group's average Gemma score")
print("New score = a * (own Gemma score) + (1 - a) * (average Gemma score of the same XL-DURel group).")
print("Label-free (uses only Gemma scores and groups), so it works identically on test data.")
print("Nested CV chooses group level, mix 'a' and cutoff inside training folds; Gemma v1 alone is an option.\n")
GRID = tuple([-1000.0] + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)] + [1000.0])
cands = [Candidate("00_gemma_v1", frozen(g), GRID)]
for t in ["0.3", "0.4", "0.5", "0.6", "0.7"]:
    gmean = defaultdict(list)
    for u in dev.usages:
        gmean[(u.word, CL[t]["|".join(u.key)])].append(g[u.key])
    gmean = {k: float(np.mean(v)) for k, v in gmean.items()}
    for a in [0.5, 0.75]:
        hyb = {u.key: a * g[u.key] + (1 - a) * gmean[(u.word, CL[t]["|".join(u.key)])] for u in dev.usages}
        cands.append(Candidate(f"hybrid_t{t}_a{a}", frozen(hyb), GRID))
print(f"Candidates: {len(cands)} (Gemma v1 + {len(cands) - 1} hybrids). Running nested CV on CPU...")
res = nested_word_cv(dev, cands)
save_evaluation(RUN, "usage_group_hybrid", res, dev, PROJECT, official=True)
pair = word_bootstrap(res["report"], other=v1)
sel = {s["test_group"][1]: s for s in res["selections"]}
wins = 0
print(f"\n{'word':<10}{'Gemma v1':>9}{'hybrid':>8}{'change':>9}   chosen by training folds")
for r in res["report"]["per_word"]:
    d = r["f1"] - gw[r["word"]]; wins += d > 0
    print(f"{r['word']:<10}{gw[r['word']]:>9.3f}{r['f1']:>8.3f}{d:>+9.3f}   {sel[r['word']]['candidate']} / cutoff {float(sel[r['word']]['threshold'])}")

adopt = pair["estimate"] >= 0.02 and wins >= 6
section("DECISION")
print(f"Hybrid minus Gemma v1 : {pair['estimate']:+.4f}  (95% interval {pair['lower']:+.4f} to {pair['upper']:+.4f})")
print(f"Words improved        : {wins}/10")
print(f"Rule (set in advance) : adopt only if gain >= +0.02 AND improved on >= 6/10 words")
print(f"\nDECISION: {'ADOPT usage-group hybrid' if adopt else 'KEEP Gemma v1'}")
save_json(RUN / "decision.json", {"corrected_ceiling": ceil, "paired_vs_v1": pair, "words_better": wins, "adopt": adopt,
                                   "note": "Corrects the earlier majority-label ceiling; tests a label-free hybrid"})
record_environment(PROJECT, RUN / "manifest.json", purpose="Corrected group ceiling + Gemma/group-mean hybrid")
print(f"\nSaved: {RUN}")